# COMP0178 — Lesson 3b: Relational Algebra

- Source: [Lecture 3b: Relational Algebra](../Lectures/Lecture_3b.pdf), all **28 PDF pages**. Use PDF page order: printed slide numbers skip values.
- Goal: read/write relational algebra, connect it to SQL, and compose operators into equivalent query plans.
- Original lecture visuals are embedded. Runnable practice and worked exam examples are labelled supplementary.
- The source calls its rental relation PropertyForRent; our reusable practice database calls it **Property**. State this naming substitution when comparing examples.

## 1. Why “relational”? Sets, tuples and domains — pages 1–6

- A **set** is unordered and contains no duplicate elements.
- A **tuple** is an ordered collection of components, e.g. `(101, "Alice")`.
- The **Cartesian product** A × B contains every pair `(a,b)` with a in A and b in B.
- A mathematical n-ary **relation** is a subset of D1 × D2 × … × Dn. Each Di is a **domain**, a set of permitted component values.
- A database table models a relation; rows are tuples, columns are named attributes associated with domains. An attribute is not literally the domain itself.
- Classical relational algebra uses **sets**, whereas SQL normally uses **bags** and may retain duplicate rows. Use DISTINCT where set semantics require it.
- Relation rows have no intrinsic display order. SQL requires ORDER BY when output order matters.

## 2. Closure and composition — pages 7–8, 15–16

- Each relational-algebra operator produces another relation. This **closure** lets one result become another operator's input.
- Read nested expressions from the inside outward. Assignment `T ← expression` names an intermediate result.
- Unary operators take one relation (selection, projection). Binary operators take two (union, difference, product, joins, division).
- Five basic operators: selection σ, projection π, union ∪, difference −, Cartesian product ×. Intersection, joins and division are derived. Renaming supports disambiguation and self-joins.

## 3. Selection σ filters rows — page 9

```text
σ[salary > 20000](Staff)
```

Keep complete Staff tuples whose salary exceeds 20,000. The square brackets here display the predicate that is often typeset as a subscript.

```sql
SELECT * FROM Staff WHERE salary > 20000;
```

Selection does not mean SQL's SELECT column list: **σ corresponds to WHERE**. Classical algebra does not have SQL NULL's three-valued logic; qualify translations involving NULL.

## 4. Projection π chooses columns — page 10

```text
π[city](Branch)
```

Keep only city, removing duplicate result tuples under set semantics:

```sql
SELECT DISTINCT city FROM Branch;
```

Combined example, read inside-out:

```text
π[staffNo, fName](σ[salary > 20000](Staff))
```

First keep qualifying rows, then keep the requested columns. Projection can reduce the number of distinct rows as well as the width.

## 5. Union, difference and compatibility — pages 11–12

| Expression | Meaning |
| --- | --- |
| R ∪ S | Tuples in either R or S, without duplicates |
| R − S | Tuples in R but not S |
| R ∩ S | Tuples present in both |

Set operands must be **union-compatible**: matching attribute structure/domains. Rename attributes when needed. SQL aligns set-operation columns by position; keep their meanings compatible too.

```text
π[city](Branch) ∪ π[city](Property)
π[city](Branch) − π[city](Property)
```

These find cities with a branch or property, and cities with a branch but no property, respectively. SQL equivalents use UNION and EXCEPT. UNION ALL keeps duplicates and is not classical set union.

## 6. Cartesian product × — pages 13–14

- R × S combines every R tuple with every S tuple.
- Cardinality: |R × S| = |R| × |S|. Degree: degree(R × S) = degree(R) + degree(S).
- Disambiguate colliding attribute names with relation qualification or renaming.
- SQL: `FROM R CROSS JOIN S` (or a comma-separated FROM without a filtering condition).
- A product alone does not match keys; an unintended missing join condition can create a huge result.

**Example:** 3 clients × 4 properties gives 12 pairs before any selection.

**Original lecture visual — PDF page 14**

[![Basic operations: selection keeps rows, projection keeps columns, and product forms all pairs.](../assets/slide-images/Lecture_3b-page-014.png)](../assets/slide-images/Lecture_3b-page-014.png)

Basic operations: selection keeps rows, projection keeps columns, and product forms all pairs.

Source: [Lecture_3b.pdf, PDF page 14](../Lectures/Lecture_3b.pdf#page=14).

## 7. Intersection and theta/equijoins — pages 17–18

```text
R ∩ S = R − (R − S)
R ⋈[condition] S = σ[condition](R × S)
```

- A **theta join** uses a comparison predicate, including =, ≠, <, ≤, > or ≥.
- An **equijoin** uses equality. Explicit predicates are helpful when column names differ or several columns share names.

```text
Client ⋈[Client.clientNo = Viewing.clientNo] Viewing
```

```sql
SELECT c.*, v.*
FROM Client c JOIN Viewing v ON c.clientNo = v.clientNo;
```

An equijoin retains both key columns unless projected away. Its algebraic definition via product does not force a physical DBMS to materialise the full product.

## 8. Natural join — page 19

- R ⋈ S automatically requires equality on **every shared attribute name**, then retains one copy of each shared column.
- If no names are shared, it is a product. If unrelated columns share a name, a natural join can impose an unintended condition.
- SQL NATURAL JOIN follows this idea; JOIN ... USING explicitly lists the matching columns and is usually clearer.
- Be careful joining Client and Staff naturally: shared fName/lName would accidentally demand equal names.

## 9. Outer joins — pages 20–21

- Inner joins return matches only.
- Left outer join keeps unmatched left rows; right outer join keeps unmatched right rows; full outer join keeps both.
- Unmatched attributes are filled with NULL. This is an **extension** beyond classical NULL-free relational algebra.
- The lecture's example joins branches and properties by city; differently named city attributes need an explicit equality predicate.
- Filtering the nullable side in WHERE can remove the unmatched rows you intended to preserve. Put conditions in ON when they define which right rows may match while retaining every left row.

## 10. Semijoin — page 22

```text
R ⋉[condition] S = π[all R attributes](R ⋈[condition] S)
```

- Keep only R's tuples that have at least one S match; return **only R's attributes**.
- Symbols vary: the lecture uses a right-pointing symbol with R retained; this notebook uses conventional left-semijoin ⋉ and explicitly states the retained side.
- SQL EXISTS is a useful equivalent. A normal SQL join may repeat an R row once per match; use EXISTS or suitable DISTINCT to match set semantics.

```sql
SELECT c.clientNo, c.fName
FROM Client c
WHERE EXISTS (SELECT 1 FROM Viewing v WHERE v.clientNo = c.clientNo);
```

Mia has two viewings but appears once. A semijoin asks “is there a match?”, not “how many matches?”.

## 11. Division ÷: “for every” — pages 23–25

For R(X,Y) and S(Y), R ÷ S returns X-values associated in R with **every Y-value in S**.

```text
R = {(Alice,A), (Alice,B), (Sam,A)}
S = {A,B}
R ÷ S = {Alice}
```

Sam is missing B. Division does not mean arithmetic division or simply “has at least one match.” Extra Y-values outside S do not disqualify a candidate.

Lecture example:

```text
π[clientNo, propertyNo](Viewing)
÷ π[propertyNo](σ[rooms = 3](PropertyForRent))
```

Find clients who viewed **all three-room properties**.

To derive division using basic operations:

```text
Candidates ← π[X](R)
Missing ← (Candidates × S) − R
Failures ← π[X](Missing)
Answer ← Candidates − Failures
```

Assume the column ordering/names are aligned for the difference. If S is empty, every candidate in π[X](R) qualifies; choosing candidates from a separate universe relation is a different query.

**Original lecture visual — PDF page 23**

[![Division removes candidate tuples that are missing a required pairing.](../assets/slide-images/Lecture_3b-page-023.png)](../assets/slide-images/Lecture_3b-page-023.png)

Division removes candidate tuples that are missing a required pairing.

Source: [Lecture_3b.pdf, PDF page 23](../Lectures/Lecture_3b.pdf#page=23).

**Original lecture visual — PDF page 24**

[![Division can be expressed using projection, product and difference.](../assets/slide-images/Lecture_3b-page-024.png)](../assets/slide-images/Lecture_3b-page-024.png)

Division can be expressed using projection, product and difference.

Source: [Lecture_3b.pdf, PDF page 24](../Lectures/Lecture_3b.pdf#page=24).

**Original lecture visual — PDF page 26**

[![Derived operations illustrated, including natural join, semijoin and division.](../assets/slide-images/Lecture_3b-page-026.png)](../assets/slide-images/Lecture_3b-page-026.png)

Derived operations illustrated, including natural join, semijoin and division.

Source: [Lecture_3b.pdf, PDF page 26](../Lectures/Lecture_3b.pdf#page=26).

## 12. Aggregation, grouping and rename — page 27

- COUNT, SUM, AVG, MIN and MAX are **extended** algebra, not among the five basic operators.
- Notation varies. The lecture uses a script aggregate symbol; these notes use γ and always specify grouping and output names.

```text
γ[branchNo; COUNT(*) → staffCount](Staff)
```

Group staff by branch and return one count per group:

```sql
SELECT branchNo, COUNT(*) AS staffCount
FROM Staff GROUP BY branchNo;
```

Without grouping, an aggregate can be represented as a one-row relation. Rename ρ gives a relation or its attributes clearer names; it is also needed to distinguish two roles of the same relation in a self-join.

```text
ρ[S](Staff)
```

SQL aliases serve a similar disambiguating purpose. NULL handling and bag/set aggregation semantics must be specified when translating SQL exactly.

**Original lecture visual — PDF page 27**

[![The lecture extends basic algebra with aggregate functions, grouping and renaming.](../assets/slide-images/Lecture_3b-page-027.png)](../assets/slide-images/Lecture_3b-page-027.png)

The lecture extends basic algebra with aggregate functions, grouping and renaming.

Source: [Lecture_3b.pdf, PDF page 27](../Lectures/Lecture_3b.pdf#page=27).

## 13. Reading a complete expression — supplementary worked example

Business question: **Which clients viewed a property with at least three rooms?**

```sql
SELECT DISTINCT c.clientNo, c.fName
FROM Client c
JOIN Viewing v ON c.clientNo = v.clientNo
JOIN Property p ON v.propertyNo = p.propertyNo
WHERE p.rooms >= 3;
```

Let C=Client, V=Viewing and P=Property, with qualified attributes:

```text
J ← (C ⋈[C.clientNo=V.clientNo] V)
     ⋈[V.propertyNo=P.propertyNo] P
Answer ← π[C.clientNo,C.fName](σ[P.rooms≥3](J))
```

It uses all three tables: names come from Client, association from Viewing, and room counts from Property. DISTINCT makes SQL projection agree with classical set semantics. A name-only projection could merge different clients with identical names.

Equivalent expression: push the Property-only filter down before joining:

```text
π[C.clientNo,C.fName](
  (C ⋈[C.clientNo=V.clientNo] V)
  ⋈[V.propertyNo=P.propertyNo] σ[P.rooms≥3](P))
```

This is valid for these inner joins. Do not blindly push conditions across outer joins, aggregates or operators that change their meaning. When pushing projection down, retain all later join and filter attributes.

## 14. Two equivalent trees — supplementary exam preparation

Each leaf is a relation. Each internal node is an operation. Read bottom-up; the root returns the final result.

```text
Tree A: filter after both joins
             π[C.clientNo,C.fName]
                       |
                  σ[P.rooms≥3]
                       |
             ⋈[V.propertyNo=P.propertyNo]
                /                    \
       ⋈[C.clientNo=V.clientNo]        P
            /         \
           C           V

Tree B: filter before the join to P
             π[C.clientNo,C.fName]
                       |
             ⋈[V.propertyNo=P.propertyNo]
                /                    \
       ⋈[C.clientNo=V.clientNo]   σ[P.rooms≥3]
            /         \                 |
           C           V                P
```

The expressions above match their respective trees. Plans can be different yet equivalent; they do not need to generate equally large intermediate results.

## 15. Runnable practice using the existing helper — supplementary

The database uses in-memory SQLite and has Branch, Staff, Client, Property and Viewing. `%%sql` runs one statement per cell. Comments-only answer cells are safe to run before solving; checks give guidance. SQL examples use portable constructs rather than requiring MySQL-specific syntax.

In [ ]:
from pathlib import Path
import sys
practice_folder = next((p for p in [Path.cwd(), Path.cwd().parent,
    Path.cwd() / "COMP0178", Path("/Users/peidi/Desktop/COMP0178")]
    if (p / "sql_practice.py").exists()), None)
if practice_folder is None:
    raise FileNotFoundError("Keep sql_practice.py in the COMP0178 folder.")
sys.path.insert(0, str(practice_folder))
from sql_practice import Practice
lab = Practice()
lab.register(get_ipython())

In [ ]:
%%sql
-- Find distinct staffNo and fName for staff with salary > 20000.
-- Order by staffNo. Write your SELECT here.

In [ ]:
lab.check([("S001", "Ada"), ("S003", "Cara")], ordered=True)

<details><summary>Practice B1 solution: selection then projection</summary>

```sql
SELECT DISTINCT staffNo, fName
FROM Staff WHERE salary > 20000 ORDER BY staffNo;
```

Algebra: π[staffNo,fName](σ[salary>20000](Staff)). Expected: S001/Ada and S003/Cara.

</details>

In [ ]:
%%sql
SELECT DISTINCT c.clientNo, c.fName
FROM Client c
JOIN Viewing v ON c.clientNo = v.clientNo
JOIN Property p ON p.propertyNo = v.propertyNo
WHERE p.rooms >= 3
ORDER BY c.clientNo;

In [ ]:
lab.check([("C001", "Mia"), ("C002", "Noah")], ordered=True)
# Additional matching viewings must not duplicate a client in the set result.
rows = lab.connection.execute("""
SELECT DISTINCT c.clientNo, c.fName FROM Client c
JOIN Viewing v ON c.clientNo=v.clientNo
JOIN Property p ON p.propertyNo=v.propertyNo
WHERE p.rooms>=3 ORDER BY c.clientNo
""").fetchall()
assert rows == [("C001", "Mia"), ("C002", "Noah")]

### Practice B2 — Division as “no required property is missing”

Find clients who viewed every property whose rooms=3. Use Client as the candidate universe. SQL's double NOT EXISTS expresses: there does not exist a required property for which no matching viewing exists.

The practice data has one such property, P002; Mia and Noah both viewed it. The empty-required-set case returns all clients under this explicitly chosen universe, unlike π[clientNo](Viewing)-restricted candidates in the literal division expression.

In [ ]:
%%sql
SELECT c.clientNo, c.fName
FROM Client c
WHERE NOT EXISTS (
    SELECT 1 FROM Property p
    WHERE p.rooms = 3
      AND NOT EXISTS (
          SELECT 1 FROM Viewing v
          WHERE v.clientNo = c.clientNo AND v.propertyNo = p.propertyNo
      )
)
ORDER BY c.clientNo;

In [ ]:
lab.check([("C001", "Mia"), ("C002", "Noah")], ordered=True)

In [ ]:
# Supplementary pure-set verification of division with more than one requirement.
R = {("Alice", "A"), ("Alice", "B"), ("Sam", "A")}
S = {"A", "B"}
candidates = {x for x, y in R}
answer = {x for x in candidates if all((x, y) in R for y in S)}
assert answer == {"Alice"}
assert {x for x in candidates if all((x, y) in R for y in set())} == candidates
print("Division result:", sorted(answer))

## 16. Past paper 2023 Q2(a–d): SQL → algebra → trees → reads

Source: [2023 paper, Q2, PDF pages 2–3](../Exampapers/COMP0178A7PE.pdf#page=2). Marks: 5 + 5 + 10 + 10. This is a worked teaching answer, not an official mark scheme.

Use the business question in section 13 and rename **Client to Clients** for the paper's exact schema. The SQL requires Clients, Viewing and Property; the expression and two trees are those in sections 13–14 with C=Clients.

The paper gives |C|=5,000, |V|=20,000, |P|=500 and no indexes. One tuple read counts as one disk read; all tables fit in memory; intermediate relations are materialised; the final result is returned directly.

**Explicit supplementary selectivity assumptions:** every viewing references exactly one client/property; 100 properties have rooms≥3; 4,000 viewings match those properties. Each operator writes a nonfinal result once and each parent rereads it once, with no reuse of a previous operator's resident output. Base inputs can remain in memory during each join, avoiding repeated scans per tuple pair. Attribute widths do not alter the paper's one-tuple-per-read convention.

| Plan | Base reads | Intermediate relations reread | Total reads | Intermediate writes | Reads + writes |
| --- | ---: | --- | ---: | ---: | ---: |
| Tree A | 25,500 | CV=20,000; CVP=20,000; selected=4,000 | 69,500 | 44,000 | 113,500 |
| Tree B | 25,500 | CV=20,000; selected P=100; joined=4,000 | 49,600 | 24,100 | 73,700 |

The root projection reads the last intermediate but does not write the final output. The 2023 question asks **reads**, so report 69,500 versus 49,600 under these assumptions, with writes separately labelled. A different explicitly justified buffering/materialisation model can change counts. The algebra does not imply nested-loop disk rescans when inputs fit in memory.

In [ ]:
base_reads = 5000 + 20000 + 500
intermediates_a = [20000, 20000, 4000]
intermediates_b = [20000, 100, 4000]
for name, sizes in [("A", intermediates_a), ("B", intermediates_b)]:
    reads = base_reads + sum(sizes)
    writes = sum(sizes)
    print(name, "reads:", reads, "writes:", writes, "accesses:", reads+writes)
assert base_reads + sum(intermediates_a) == 69500
assert base_reads + sum(intermediates_b) == 49600

## 17. Past paper 2024 Q2: all three surgery tables

Source: [2024 Q2(a–c), PDF pages 2–3](../Exampapers/COMP0178A7PF.pdf#page=2). Printed parts are 5+5+10=20, but the question total says 30; this discrepancy is in the supplied paper.

**Business question:** which patient and physician IDs/names occur in appointments in room 3A on the chosen date? All three tables are necessary.

```sql
SELECT DISTINCT p.patientID, p.lName, d.physicianID, d.lName
FROM Patients p
JOIN Appointments a ON p.patientID = a.patientID
JOIN Physicians d ON a.physicianID = d.physicianID
WHERE a.room = '3A' AND a.date = '2024-05-14';
```

The date is a fixed practice parameter, not a claim about today's date. Let P=Patients, A=Appointments, D=Physicians; predicate q is room/date above and L is the four projected attributes.

```text
Tree/expression A:
π[L](σ[q]((P ⋈[P.patientID=A.patientID] A)
                ⋈[A.physicianID=D.physicianID] D))

Tree/expression B:
π[L]((P ⋈[P.patientID=A.patientID] σ[q](A))
                ⋈[A.physicianID=D.physicianID] D)

A: π → σq → join-D → join-P-A → leaves
B: π → join-D → join-P-filteredA → σq on A
```

The nesting specifies the tree exactly: join nodes have the two operands shown in parentheses. Drawing the corresponding branches follows section 14's pattern.


**Corresponding trees:** J_PA means P.patientID=A.patientID; J_AD means A.physicianID=D.physicianID. q is the room/date predicate, and πL retains the four named output attributes.

```text
Plan A                       Plan B
       πL                           πL
        |                            |
       σq                          J_AD
        |                         /    \
      J_AD                      J_PA    D
     /    \                    /    \
   J_PA    D                  P      σq
  /    \                              |
 P      A                             A
```

**Worked costs under section 16's materialise-and-reread convention:** base rows P=15,000, A=40,000, D=8 give 55,008 base reads. Assume all FK matches exist. The supplied 20 appointments/physician/day gives about 160/day; additionally assume 16 are in room 3A. These room-selectivity and date-validity assumptions are explicit.

- Plan A intermediates: 40,000 + 40,000 + 16. Reads=135,024; writes=80,016; total accesses=215,040.
- Plan B intermediates: 16 + 16 + 16. Reads=55,056; writes=48; total accesses=55,104.

Each listed intermediate is written once and reread once. The final projection is returned without a write. Do not silently count accesses as reads alone.

## 18. Past paper 2025 Q2: use a different business question

Source: [2025 Q2(a–d), PDF pages 2–4](../Exampapers/COMP0178A7PG.pdf#page=2). The paper explicitly requires a question different from its suggested quantity-by-state example.

**Our different question:** which customers ordered item 323? Return customer IDs and names, not quantities grouped by state.

```sql
SELECT DISTINCT c.custID, c.firstname, c.familyname
FROM Customers c
JOIN Orders o ON c.custID = o.custID
JOIN LineItems l ON o.orderID = l.orderID
WHERE l.itemID = 323;
```

Let C=Customers, O=Orders, L=LineItems, q be L.itemID=323, and N be the three output attributes.

```text
A = π[N](σ[q]((C ⋈[C.custID=O.custID] O)
                    ⋈[O.orderID=L.orderID] L))
B = π[N]((C ⋈[C.custID=O.custID] O)
                    ⋈[O.orderID=L.orderID] σ[q](L))
```

Draw A with selection above both joins; draw B with selection directly above L. This is the same two-tree transformation as section 14, with the exact predicates shown here.

**Worked cost assumptions:** base sizes C=3,000, O=6,000, L=10,000; only those three tables are queried, so do not read the 600-row Items table. Assume valid FKs and 100 matching line items. Use materialise-and-reread with in-memory join inputs as before.

- Base reads = 19,000.
- A intermediates: CO=6,000; COL=10,000; filtered=100. Reads=35,100; writes=16,100; total accesses=51,200.
- B intermediates: CO=6,000; filtered L=100; joined=100. Reads=25,200; writes=6,200; total accesses=31,400.

Projection may deduplicate customers; its output is not materialised, so no final-write cost is added. These are assumption-labelled worked answers, not uniquely determined counts from unspecified selectivities.

## 19. Common mistakes and revision — page 28

- σ filters **rows**; π chooses **columns**.
- Projection in classical algebra removes duplicates; SQL SELECT ordinarily does not.
- A product is all pairs; a join adds a matching condition.
- A natural join matches every shared name, not just the key you had in mind.
- An outer join preserves unmatched rows; a later WHERE can remove them.
- A semijoin returns attributes from the retained side, without one copy per matching partner.
- Division means **every required value**, not any one value.
- Aggregates need extended algebra; ordinary π/σ alone do not express COUNT.
- Push filters down only when equivalent, and retain join keys when pushing projections.
- Keep logical algebra separate from physical execution and storage-cost assumptions.

### Revision questions with answers

1. Which operation corresponds to WHERE? Selection σ.
2. Does π[city](Branch) keep duplicate cities? No, under set semantics.
3. How large is a 5-row × 8-row product? 40 rows, before filtering.
4. R∩S using difference only? R−(R−S).
5. What does a semijoin return? Matching tuples from the designated retained side only.
6. What does division test? Coverage of every required right-side tuple.
7. Why DISTINCT in the three-table example? Several viewings can lead to the same projected client.
8. Is a cached pointer traversal automatically a disk read? No.
9. Can different trees give the same answer? Yes; equivalent logical plans can have different intermediate sizes.
10. What must an exam cost answer state? Table/cardinality assumptions, selection estimates, buffering/materialisation rules, reads versus writes and final-result treatment.